In [ ]:
from sentence_transformers import SentenceTransformer
import torch
# Load embedding model
device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer("google/embeddinggemma-300m", device=device)

def create_embedding(texts):
    if type(texts) == list: # the input param is a list contains text chunks
        embeddings = model.encode(texts)
        return embeddings
    elif type(texts) == str: # just one text chunk
        embedding = model.encode([texts])
        return embedding
    else:
        raise ValueError("The texts paramters should be str or list[str]")

print(device)

In [ ]:
import os
dir_path = os.getcwd()
print("The directory of this script is:", dir_path)
root_path = os.path.dirname(dir_path)
print("The root directory is:", root_path)

In [ ]:
#LLM api call
from google import genai
with open(f"{root_path}/API_KEY.txt", "r", encoding="utf-8") as f:
    API_KEY = f.read()

def call_gemini(text):
    client = genai.Client(api_key = API_KEY)
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=text
    )
    return response.text, response.usage_metadata.total_token_count

In [ ]:
import sys
import numpy as np
sys.path.append(root_path)
from testing.metrics.faithfulness import compute_faithfulness #Detect hallucinations: Measures how much the LLM answer is based on the retrieved context
"""
question: str,
answer: str,
contexts: List[str],
call_gemini,
max_retries: int = 5
"""
from testing.metrics.context_recall import compute_context_recall #Evaluate retrieval coverage: Measures how much of the reference (ground-truth) answer is supported by the retrieved context.
"""
question: str,
contexts: List[str],
reference_answer: str,
call_gemini,
max_retries: int = 5
"""
from testing.metrics.coverage import compute_coverage #Measures how completely a model’s response covers the factual content of a reference (ground-truth) answer.
"""
question: str,
reference: str,
response: str,
call_gemini,
max_retries: int = 5
"""
from testing.metrics.rouge import compute_rouge
#Calculate F-1 of lexical overlap by LCS: longest common subsequence (in terms of word order, adjacency not required)
#Precision = len(LCS) / len(Answer)
#Recall = len(LCS) / len(Ref)
"""
answer: str,
ground_truth: str,
rouge_type: str = "rougeL",
mode: str = "fmeasure"
"""
from testing.metrics.accuracy import compute_answer_accuracy
"""
question: str,
answer: str,
ground_truth: str,
call_gemini,
create_embedding,
weights: List[float] = [0.75, 0.25],
beta: float = 1.0,
max_retries: int = 5
"""
print("Import evaluation functions")


In [ ]:
import pandas as pd
medical_questions_answered_loaded = pd.read_parquet("data/medical_questions_answered.parquet")
relevant_questions = medical_questions_answered_loaded[medical_questions_answered_loaded["question_type"] == "Contextual Summarize"]
question_ids = relevant_questions['id'].to_list()
relevant_questions

In [ ]:
try:
    medical_questions_evaluated = pd.read_parquet("data/evaluations/contextual_summarization_evaluated.parquet")
    medical_questions_evaluated = (
        medical_questions_evaluated
        .set_index("qid")
        .to_dict(orient="index")
    )
except:
    medical_questions_evaluated = {}


medical_questions_evaluated

In [ ]:
def all_valid(values):
    return all(
        v is not None and not np.isnan(v)
        for v in values
    )

In [ ]:
#accuracy
#coverage

from tqdm import tqdm
import time
sep = "\n\n" + "-"*100 + "\n\n"
MAX_RETRIES = 10
SLEEP_SEC = 30

def get_answers():
    for qid in tqdm(question_ids):
        if qid in medical_questions_evaluated:
            continue

        row = relevant_questions.loc[relevant_questions["id"] == qid].iloc[0]
        row_eval = {}

        question = row["question"]
        reference = row["answer"]
        answer = row["LLM_answer"]
        context = row["LLM_context"]
        contexts = context.split(sep)
        
        accuracy, token1 = None, None
        for attempt in range(1, MAX_RETRIES + 1):
            try:
                accuracy, token1 = compute_answer_accuracy(question, answer, reference, call_gemini, create_embedding)
                break
            except Exception as e:
                code = e.error.code if hasattr(e, "error") else e
                print(f"[ERROR] qid={qid} accuracy calculation failed after {attempt} retries: {code}")
                if attempt == MAX_RETRIES:
                    return
                time.sleep(SLEEP_SEC * attempt)  #backoff

        coverage, token2 = None, None
        for attempt in range(1, MAX_RETRIES + 1):
            try:
                coverage, token2 = compute_coverage(question, reference, answer, call_gemini)
                break
            except Exception as e:
                code = e.error.code if hasattr(e, "error") else e
                print(f"[ERROR] qid={qid} coverage calculation failed after {attempt} retries: {code}")
                if attempt == MAX_RETRIES:
                    return
                time.sleep(SLEEP_SEC * attempt)  #backoff
        if all_valid([accuracy, coverage, token1, token2]):
            row_eval["accuracy"] = accuracy
            row_eval["coverage"] = coverage
            row_eval["tokens"] = token1 + token2
            medical_questions_evaluated[qid] = row_eval
        break
get_answers()


In [ ]:
medical_questions_evaluated

In [ ]:
df_eval.to_parquet("data/evaluations/contextual_summarization_evaluated.parquet")
df_eval_loaded = pd.read_parquet("data/evaluations/contextual_summarization_evaluated.parquet")
df_eval_loaded

In [ ]:
print("min:", df_eval_loaded["tokens"].min())
print("max:", df_eval_loaded["tokens"].max())
print("avg:", df_eval_loaded["tokens"].mean())
print("sum:", df_eval_loaded["tokens"].sum())